# 04 — Local Outlier Factor (LOF)

## 1. Intuition

The main idea behind **Local Outlier Factor** is different from Isolation Forest and One-Class SVM.

Instead of asking:

> **"Can I isolate this point quickly?"** — Isolation Forest

or:

> **"Is this point inside the normal boundary?"** — One-Class SVM

LOF asks:

> **"Is this point much less dense than the points around it?"**

This is the key idea.

---

## Step 1: Imagine a dense group

Suppose we have these points:

```text id="e3l4k7"
      ● ● ●
    ● ● ● ● ●
    ● ● ● ●
      ● ●
```

The points are close together.

So the area has **high local density**.

---

## Step 2: Now add an isolated point

```text id="xq1q3m"
      ● ● ●
    ● ● ● ● ●
    ● ● ● ●
      ● ●


                         ×
```

The point `×` is far away from the other observations.

Its **local neighborhood is sparse**.

LOF recognizes this difference.

---

# Step 3: Why "local" is important

This is the most important part of LOF.

Consider this dataset:

```text id="2u1x8a"
Dense group A                 Sparse group B

● ● ● ●                       ●
● ● ●                         ●
● ● ● ●                       ●
```

Suppose we have one point:

```text id="7p4yqv"
● ● ● ●                       ●
● ● ●                         ●
● ● ● ●                       ●
```

The second group is naturally more spread out.

A **global density method** might say:

> "Group B is sparse, therefore its points are anomalies."

But that's not necessarily true.

LOF instead asks:

> **"Is this point's density significantly lower than the density of its own neighbors?"**

So if all points around it are also sparse:

```text
●     ●
    ●
●     ●
```

then that point may still be considered **normal**.

But if one point is much more isolated than its neighbors:

```text
● ● ● ●
● ● ● ●
● ● ● ●


                 ×
```

then LOF considers it suspicious.

---

# Step 4: The core comparison

LOF essentially compares:

$$
\boxed{\text{Density of the point}}
$$

with:

$$
\boxed{\text{Density of its neighbors}}
$$

If:

$$
\text{density(point)}
\approx
\text{density(neighbors)}
$$

→ likely normal.

If:

$$
\text{density(point)}
\ll
\text{density(neighbors)}
$$

→ likely anomaly.

So the fundamental idea is:

```text id="z6e9kj"
Point
  ↓
Find nearby neighbors
  ↓
Measure local density
  ↓
Measure neighbors' local density
  ↓
Compare them
  ↓
Much lower density?
  ↓
Anomaly
```

---

## Step 5: Why not simply use distance?

You might think:

> "If a point is far from other points, just call it an anomaly."

But that doesn't always work.

Imagine:

```text id="d5d8ae"
Cluster A                 Cluster B

● ● ●                    ●       ●
● ● ●                    ●   ●   ●
● ● ●                    ●       ●
```

Cluster B is naturally more spread out.

A point in Cluster B may have larger distances from its neighbors than a point in Cluster A, but it isn't necessarily anomalous.

LOF handles this by looking at **relative local density**, rather than only absolute distance.

That's why the word **Local** is so important.

---

# The mental model

Keep this chain in mind throughout the LOF notebook:

```text id="0f5m6r"
Data points
     ↓
Find k-nearest neighbors
     ↓
Calculate local density
     ↓
Compare point's density
with neighbors' densities
     ↓
LOF score
     ↓
Normal / Anomaly
```

The upcoming concepts will build this chain step-by-step:

**k-Nearest Neighbors → Reachability Distance → Local Reachability Density → LOF Score → Anomaly Decision.**

**Question:** So instead of considering one global distance between points, we consider only the local distance, like only that cluster?

**Answer:** **Yes, essentially.** LOF focuses on the **local neighborhood around each point** rather than comparing everything using one global distance/density measure.

Small correction: LOF does **not need to identify the cluster first**. It finds the nearby points using **k-nearest neighbors** and compares the point's local density with the density of those neighbors.

So:

> **Global approach:** Consider the whole dataset's density.
> **LOF:** Consider the point's **local neighborhood** and compare it with nearby points.


# 2. k-Nearest Neighbors (k-NN)

Now that we understand **why LOF looks locally**, the next question is:

> **How does LOF decide which points belong to a point's local neighborhood?**

It uses **k-Nearest Neighbors**.

---

## 1. What is a neighbor?

Suppose we have these points:

```text
      A ●

  B ●       C ●

        X ●

              D ●
```

For point \(X\), its neighbors are the points that are **closest to X**.

For example, if the distances are:

| Point | Distance from X |
| ----- | --------------: |
| A     |               2 |
| B     |               3 |
| C     |               1 |
| D     |               5 |

Then the nearest points are:

```text
C → 1
A → 2
B → 3
D → 5
```

---

## 2. What does \(k\) mean?

\(k\) simply tells us **how many nearest neighbors we want to consider**.

If:

$$
k=2
$$

we take the **2 closest points**.

From our example:

```text
X
│
├── C → distance 1
└── A → distance 2
```

So:

$$
N_k(X)=\{C,A\}
$$

where \(N_k(X)\) means the **k-nearest neighbors of \(X\)**.

---

## 3. Why does LOF need this?

Because LOF wants to measure **local density**.

To measure the local density around \(X\), it first needs to know:

> "Which points are considered close to \(X\)?"

That's exactly what k-NN provides.

So the process starts:

```text
Point X
   ↓
Find k nearest neighbors
   ↓
Measure distances
   ↓
Use those neighbors to estimate local density
```

---

## 4. Small LOF example

Suppose:

```text
      ● ● ●
    ● ● X ● ●
      ● ● ●
```

If:

$$
k=4
$$

LOF might select the four closest points around \(X\):

```text
      ●
    ● X ●
      ●
```

Then it uses these four neighbors to understand the density around \(X\).

Now suppose another point is far away:

```text
      ● ● ●
    ● ● ● ● ●


                         X
```

Its 4 nearest neighbors might all be relatively far away.

That tells LOF:

> The local neighborhood around \(X\) is sparse.

But remember, **sparse alone doesn't automatically mean anomaly**. LOF will later compare this density with the densities of \(X\)'s neighbors.

---

## 5. Why choosing \(k\) matters

Suppose:

$$
k=2
$$

LOF looks at a **very small neighborhood**.

This makes it sensitive to very local changes.

If:

$$
k=20
$$

LOF considers a much larger neighborhood.

This gives a broader view of the surrounding data.

So:

```text
Small k → very local view
Large k → broader local view
```

We'll discuss the practical choice of `n_neighbors` when we reach the hyperparameters.

---

## The important chain

We're building LOF step-by-step:

$$
\boxed{
\text{Point}
\rightarrow
\text{k nearest neighbors}
\rightarrow
\text{local density}
}
$$


## How do we select \(k\) in LOF?

There is **no single correct value** of \(k\). It depends on how local you want the anomaly detection to be.

### 1. Small \(k\)

Example:

$$
k=3
$$

LOF looks at only the **3 closest points**.

```text
X
● ● ●
```

Good for detecting **very local anomalies**, but it can be sensitive to noise.

---

### 2. Large \(k\)

Example:

$$
k=20
$$

LOF looks at 20 neighbors.

```text
X
● ● ● ● ● ● ● ● ●
      ...more...
```

It gets a **broader view**, but very local anomalies may become less obvious.

---

### 3. Practical approach

Usually, you choose \(k\) based on:

* **Dataset size**
* **Expected size of local groups**
* **How local the anomalies are**
* Experimenting with several values and checking whether the detected anomalies are stable.

For example:

```python
from sklearn.neighbors import LocalOutlierFactor

model = LocalOutlierFactor(n_neighbors=20)
```

`20` is a common starting point, **not a universal rule**.

### Most important idea

> **\(k\) controls the size of the neighborhood LOF uses to judge whether a point is locally unusual.**

$$
\boxed{\text{Small }k \rightarrow \text{more local}}
$$

$$
\boxed{\text{Large }k \rightarrow \text{broader neighborhood}}
$$

Also, \(k\) should be **smaller than the number of observations** in your dataset.


# 3. Reachability Distance

Now we move from **finding the neighbors** to understanding **how LOF measures distance for density calculation**.

You might think LOF simply uses:

$$
d(A,B)
$$

the normal distance between two points.

But LOF uses something slightly different called **reachability distance**.

---

## 1. Why not use ordinary distance?

Consider this:

```text id="q4y8gk"
A ●────● B────● C
```

Suppose:

$$
d(A,B)=2
$$

and:

$$
d(B,C)=2
$$

That's straightforward.

But imagine the data is uneven:

```text id="xwq2i8"
A ●──● B────────────● C
```

Here:

$$
d(A,B)=2
$$

but:

$$
d(B,C)=10
$$

Using raw distance can make density calculations unstable when some regions contain points that are naturally spread out.

LOF therefore uses a **smoothed distance** called reachability distance.

---

# 2. First understand k-distance

Before defining reachability distance, we need one more concept:

### k-distance

For a point \(B\), its **k-distance** is the distance from \(B\) to its \(k\)-th nearest neighbor.

Suppose:

$$
k=3
$$

and the distances from \(B\) to its neighbors are:

$$
2,\;4,\;7,\;10
$$

The 3rd nearest neighbor is at distance:

$$
\boxed{k\text{-distance}(B)=7}
$$

So:

> **k-distance tells us how far we need to go from a point to reach its \(k\)-th nearest neighbor.**

---

# 3. Reachability distance

Now we can define it.

For two points \(A\) and \(B\):

$$
\boxed{
\operatorname{reach-dist}_k(A,B)
=
\max
\left(
k\text{-distance}(B),
d(A,B)
\right)
}
$$

Notice something important:

> We use the **k-distance of \(B\)**, not \(A\).

---

## 4. Why take the maximum?

Let's use numbers.

Suppose:

$$
k\text{-distance}(B)=5
$$

and:

$$
d(A,B)=2
$$

Then:

$$
\operatorname{reach-dist}_k(A,B)
=
\max(5,2)
$$

$$
=5
$$

Even though A and B are only 2 units apart, LOF uses **5**.

---

### Another example

Suppose:

$$
k\text{-distance}(B)=5
$$

and:

$$
d(A,B)=8
$$

Then:

$$
\operatorname{reach-dist}_k(A,B)
=
\max(5,8)
$$

$$
=8
$$

So:

```text id="q0v8cz"
Normal distance = 2
k-distance      = 5

Reachability distance = 5
```

and:

```text id="3h2w6j"
Normal distance = 8
k-distance      = 5

Reachability distance = 8
```

---

# 5. Why does LOF do this?

The purpose is to **avoid overestimating density in very dense regions**.

Imagine:

```text id="7m7o3b"
●●●●●
```

Points are extremely close together.

If we used only raw distances, some distances could be extremely tiny.

That could make the calculated density extremely large.

Reachability distance introduces a kind of **local distance floor** based on the neighborhood size.

So instead of saying:

> "These two points are only 0.1 apart, therefore the density is enormously high."

LOF uses the neighborhood structure to provide a more stable density calculation.

---

# 6. Connecting it to what we already learned

We have now built:

```text id="t3aq5b"
Point
  ↓
Find k nearest neighbors
  ↓
Find k-distance
  ↓
Calculate reachability distance
  ↓
Use these distances to calculate local density
```

And the formula to remember is:

$$
\boxed{
\operatorname{reach-dist}_k(A,B)
=
\max
\left(
k\text{-distance}(B),
d(A,B)
\right)
}
$$

### The key intuition

> **Reachability distance is a modified distance that prevents extremely small local distances from making density estimates unrealistically large.**

The next step is **Local Reachability Density (LRD)** — this is where these reachability distances are actually converted into a **density value**.


**Question:** How is this reachability distance actually used?

**Answer:** The reachability distance is used to calculate the **local density of a point**. This is the next step in LOF.

Let's use a tiny example.

Suppose \(A\) has 3 neighbors:

```text
A → B
A → C
A → D
```

After calculating reachability distances:

$$
\operatorname{reach-dist}(A,B)=2
$$

$$
\operatorname{reach-dist}(A,C)=3
$$

$$
\operatorname{reach-dist}(A,D)=4
$$

LOF takes their **average**:

$$
\frac{2+3+4}{3}=3
$$

This average reachability distance tells us roughly:

> **How far A is from the points in its local neighborhood.**

Then LOF converts that into **density**:

$$
\boxed{
LRD(A)=
\frac{1}
{\text{average reachability distance of A}}
}
$$

So here:

$$
LRD(A)=\frac{1}{3}=0.333
$$

---

### Why inverse?

Because **smaller distance means higher density**.

For example:

$$
\text{Average distance}=2
\Rightarrow LRD=0.5
$$

$$
\text{Average distance}=10
\Rightarrow LRD=0.1
$$

So:

```text
Small reachability distance
        ↓
High density

Large reachability distance
        ↓
Low density
```

And this is exactly what LOF needs.

It can now compare:

```text
Density of X
     vs
Density of X's neighbors
```

If X has **much lower density** than its neighbors → X is likely an anomaly.

So the complete chain is:

$$
\boxed{
\text{k-NN}
\rightarrow
\text{Reachability Distance}
\rightarrow
\text{Local Density}
\rightarrow
\text{Compare Densities}
\rightarrow
\text{LOF Score}
}
$$


# 4. Local Reachability Density (LRD)

Now we take the **reachability distances** we calculated and convert them into a measure of **local density**.

The idea is simple:

> If a point is close to its neighbors, its local density is high.
> If a point is far from its neighbors, its local density is low.

---

## 1. Calculate reachability distances

Suppose point \(A\) has 3 nearest neighbors:

$$
B,\ C,\ D
$$

and their reachability distances from \(A\) are:

$$
2,\ 3,\ 4
$$

We calculate their average:

$$
\frac{2+3+4}{3}=3
$$

So the average reachability distance of \(A\) is:

$$
3
$$

---

## 2. Convert distance into density

Density should behave opposite to distance:

```text
Small distance → High density
Large distance → Low density
```

So LOF uses the reciprocal:

$$
\boxed{
LRD(A)=
\frac{1}
{\text{average reachability distance}}
}
$$

For our example:

$$
LRD(A)=\frac{1}{3}
$$

$$
\boxed{LRD(A)\approx0.333}
$$

---

## 3. Compare two points

Suppose we have:

### Point A

Reachability distances:

$$
2,\ 3,\ 4
$$

Average:

$$
3
$$

Therefore:

$$
LRD(A)=\frac{1}{3}=0.333
$$

### Point X

Reachability distances:

$$
8,\ 10,\ 12
$$

Average:

$$
10
$$

Therefore:

$$
LRD(X)=\frac{1}{10}=0.1
$$

Now compare:

```text
A → LRD = 0.333 → relatively dense
X → LRD = 0.100 → relatively sparse
```

So \(X\)'s local region is much less dense.

---

## 4. Actual LOF formula

The formal definition is:

$$
\boxed{
LRD_k(A)=
\frac{|N_k(A)|}
{\displaystyle\sum_{B\in N_k(A)}
\operatorname{reach-dist}_k(A,B)}
}
$$

where:

* \(N_k(A)\) = the \(k\)-nearest neighbors of \(A\)
* \(|N_k(A)|\) = number of neighbors
* \(\operatorname{reach-dist}_k(A,B)\) = reachability distance between \(A\) and neighbor \(B\)

If \(k=3\), then:

$$
LRD_k(A)
=
\frac{3}
{2+3+4}
=
\frac{3}{9}
=
0.333
$$

This is mathematically the same as:

$$
\frac{1}{\text{average reachability distance}}
$$

---

## 5. Why LRD is important

Now every point has a **local density value**.

For example:

| Point |   LRD |
| ----- | ----: |
| A     | 0.333 |
| B     | 0.300 |
| C     | 0.350 |
| X     | 0.100 |

We can now see that \(X\) has a much lower local density than the other points.

But **low density alone isn't enough**.

A whole sparse cluster could have low LRD values:

```text
Dense cluster          Sparse cluster

●●●                    ●       ●
●●●                    ●   ●   ●
●●●                    ●       ●
```

The sparse cluster's points may all have low LRD, but they can still be normal because their **neighbors also have similarly low density**.

That's why LOF ultimately compares a point's LRD with the LRDs of its neighbors.

$$
\boxed{
\text{LRD}
\rightarrow
\text{Density of the point's local neighborhood}
}
$$


**Question:** So for a point, LRD is calculated once, then LRD is calculated for all its neighbors, and then we see how dense or sparse the neighborhood is?

**Answer:** **Yes, exactly.** You're understanding it correctly.

For a point \(A\):

```text
             A
             ↓
      Find k neighbors
             ↓
    ┌────────┼────────┐
    ↓        ↓        ↓
    B        C        D
    ↓        ↓        ↓
   LRD      LRD      LRD
```

We calculate:

1. **LRD of \(A\)** → density around \(A\)
2. **LRD of each neighbor** \(B,C,D\) → density around each neighbor
3. Then compare \(A\)'s density with the densities of its neighbors.

For example:

$$
LRD(A)=0.10
$$

while:

$$
LRD(B)=0.30,\quad
LRD(C)=0.25,\quad
LRD(D)=0.35
$$

Then \(A\) is much less dense than its neighbors:

```text
A's density        = 0.10  ← sparse
Neighbors' density ≈ 0.30  ← dense
```

Therefore, \(A\) is potentially an **outlier**.

The important correction is:

> We aren't just asking whether \(A\)'s neighborhood is sparse. We ask whether **\(A\)'s neighborhood is sparse compared with the neighborhoods of \(A\)'s neighbors**.

That comparison is what produces the **LOF score**.


# 5. LOF Score

Now we have the two things we need:

* **LRD of the point**
* **LRD of its neighbors**

LOF compares them.

## 1. The basic idea

Suppose we're checking point \(A\).

We calculated:

$$
LRD(A)=0.10
$$

and for its neighbors:

$$
LRD(B)=0.30
$$

$$
LRD(C)=0.25
$$

$$
LRD(D)=0.35
$$

The neighbors are much denser than \(A\).

So \(A\) should get a **high LOF score**.

---

## 2. Formula

The LOF score is:

$$
\boxed{
LOF_k(A)=
\frac{1}{|N_k(A)|}
\sum_{B\in N_k(A)}
\frac{LRD_k(B)}
{LRD_k(A)}
}
$$

In simple words:

> **LOF = average density of the neighbors ÷ density of the point**

---

## 3. Calculate our example

We have:

$$
LRD(A)=0.10
$$

Neighbors:

$$
0.30,\quad0.25,\quad0.35
$$

So:

$$
LOF(A)
=
\frac{1}{3}
\left(
\frac{0.30}{0.10}
+
\frac{0.25}{0.10}
+
\frac{0.35}{0.10}
\right)
$$

Calculate each:

$$
\frac{0.30}{0.10}=3
$$

$$
\frac{0.25}{0.10}=2.5
$$

$$
\frac{0.35}{0.10}=3.5
$$

Therefore:

$$
LOF(A)=
\frac{3+2.5+3.5}{3}
$$

$$
=\frac{9}{3}
$$

$$
\boxed{LOF(A)=3}
$$

---

## 4. How do we interpret the score?

The most important reference point is:

$$
\boxed{LOF\approx1}
$$

### LOF ≈ 1

The point has approximately the **same density as its neighbors**.

```text
Point density ≈ Neighbor density
             ↓
           LOF ≈ 1
             ↓
        likely normal
```

### LOF > 1

The point is **less dense than its neighbors**.

```text
Point density < Neighbor density
             ↓
          LOF > 1
             ↓
      potentially anomalous
```

### LOF much greater than 1

The point is **much less dense** than its neighbors.

```text
Point density << Neighbor density
             ↓
        LOF >> 1
             ↓
       strong outlier
```

---

## 5. What about LOF < 1?

Suppose:

$$
LRD(A)=0.30
$$

but its neighbors have:

$$
LRD\approx0.10
$$

Then \(A\) is actually **denser than its neighbors**.

Its LOF can be below 1.

That doesn't mean "very normal" in a probability sense. It simply means the point is **denser than its local neighbors**.

---

## 6. The complete LOF process

We can now connect everything:

```text
Point A
   ↓
Find k nearest neighbors
   ↓
Calculate reachability distances
   ↓
Calculate LRD(A)
   ↓
Calculate LRD of A's neighbors
   ↓
Compare their densities
   ↓
Calculate LOF(A)
   ↓
Interpret score
```

The central idea is:

$$
\boxed{
LOF =
\frac{\text{local density of neighbors}}
{\text{local density of point}}
}
$$

So if the point's density is much smaller than its neighbors' density, the denominator is small and the LOF becomes large.

That's why **a high LOF indicates a local outlier**.


# 6. Numerical Example — LOF

Let's now calculate LOF **from beginning to end** for a small dataset.

We'll use **one-dimensional data** first so the calculations are easy to follow.

Suppose:

$$
X=
\begin{bmatrix}
1\\
2\\
3\\
4\\
5\\
10
\end{bmatrix}
$$

Visually:

```text id="4l8p3n"
1   2   3   4   5                 10
●   ●   ●   ●   ●                  ×
```

We suspect that \(10\) might be an anomaly.

Let's use:

$$
k=2
$$

So each point will consider its **2 nearest neighbors**.

---

## Step 1: Find the neighbors of \(10\)

Distances from \(10\):

$$
d(10,5)=5
$$

$$
d(10,4)=6
$$

$$
d(10,3)=7
$$

The two closest points are:

$$
N_2(10)=\{5,4\}
$$

So:

```text id="v8t2q1"
10
│
├── 5  → distance 5
└── 4  → distance 6
```

---

## Step 2: Calculate k-distance

For point \(10\), its 2nd nearest neighbor is \(4\).

Therefore:

$$
k\text{-distance}(10)=6
$$

For point \(5\), its two nearest neighbors are \(4\) and \(3\):

$$
d(5,4)=1
$$

$$
d(5,3)=2
$$

Therefore:

$$
k\text{-distance}(5)=2
$$

For point \(4\), its two nearest neighbors are \(3\) and \(5\):

$$
d(4,3)=1
$$

$$
d(4,5)=1
$$

Therefore:

$$
k\text{-distance}(4)=1
$$

---

## Step 3: Calculate reachability distances for \(10\)

Remember:

$$
\operatorname{reach-dist}_k(A,B)
=
\max
\left(
k\text{-distance}(B),
d(A,B)
\right)
$$

For \(10\) and \(5\):

$$
\operatorname{reach-dist}_2(10,5)
=
\max(2,5)
=5
$$

For \(10\) and \(4\):

$$
\operatorname{reach-dist}_2(10,4)
=
\max(1,6)
=6
$$

So:

```text id="u5r8am"
10 → neighbor 5 → reachability distance = 5
10 → neighbor 4 → reachability distance = 6
```

---

## Step 4: Calculate LRD of \(10\)

`"k" in numerator below is the number of neighbours considered`
The formula is:

$$
LRD_k(10)
=
\frac{k}
{\sum_{B\in N_k(10)}
\operatorname{reach-dist}_k(10,B)}
$$

Since \(k=2\):

$$
LRD_2(10)
=
\frac{2}{5+6}
$$

$$
LRD_2(10)
=
\frac{2}{11}
$$

$$
\boxed{LRD_2(10)\approx0.182}
$$

So \(10\) has a **low local density**.

---

# Step 5: Calculate LRD of its neighbors

Now remember what you correctly understood earlier:

> We also need the LRD of \(10\)'s neighbors.

So we need:

$$
LRD(5)
$$

and:

$$
LRD(4)
$$

### LRD of 5

Its neighbors are \(4\) and \(3\).

Reachability distances:

$$
\operatorname{reach-dist}(5,4)
=
\max(k\text{-distance}(4),d(5,4))
$$

$$
=\max(1,1)=1
$$

And:

$$
\operatorname{reach-dist}(5,3)
=
\max(k\text{-distance}(3),d(5,3))
$$

For this small example, \(k\text{-distance}(3)=2\), so:

$$
=\max(2,2)=2
$$

Therefore:

$$
LRD(5)
=
\frac{2}{1+2}
=
\frac23
$$

$$
\boxed{LRD(5)\approx0.667}
$$

---

### LRD of 4

Its neighbors are \(3\) and \(5\).

The reachability distances are:

$$
\operatorname{reach-dist}(4,3)=2
$$

and:

$$
\operatorname{reach-dist}(4,5)=1
$$

Therefore:

$$
LRD(4)
=
\frac{2}{2+1}
$$

$$
\boxed{LRD(4)\approx0.667}
$$

---

# Step 6: Calculate LOF of 10

Now we have everything:

$$
LRD(10)=0.182
$$

$$
LRD(5)=0.667
$$

$$
LRD(4)=0.667
$$

LOF formula:

$$
LOF_k(10)
=
\frac{1}{k}
\sum_{B\in N_k(10)}
\frac{LRD_k(B)}
{LRD_k(10)}
$$

Therefore:

$$
LOF_2(10)
=
\frac12
\left(
\frac{0.667}{0.182}
+
\frac{0.667}{0.182}
\right)
$$

Each ratio is approximately:

$$
\frac{0.667}{0.182}\approx3.67
$$

Therefore:

$$
LOF_2(10)
=
\frac{3.67+3.67}{2}
$$

$$
\boxed{LOF_2(10)\approx3.67}
$$

---

# Step 7: Interpret it

We got:

$$
LOF(10)\approx3.67
$$

Remember:

$$
LOF\approx1
$$

means the point has approximately the same density as its neighbors.

But:

$$
LOF(10)\approx3.67
$$

means \(10\)'s local density is **much lower** than the density around its neighbors.

So:

$$
\boxed{10\rightarrow\text{strong local outlier}}
$$

---

## The complete calculation

```text id="b0a5qh"
Dataset
1  2  3  4  5  10
                  ↓
              Check 10
                  ↓
        Find 2 nearest neighbors
                  ↓
               5, 4
                  ↓
        Calculate reachability
             distances
                  ↓
               5, 6
                  ↓
             Calculate LRD
                  ↓
             LRD(10)=0.182
                  ↓
       Calculate LRD of neighbors
                  ↓
       LRD(5)=0.667, LRD(4)=0.667
                  ↓
             Calculate LOF
                  ↓
              LOF(10)=3.67
                  ↓
            Local Outlier
```

This is the central calculation behind LOF:

$$
\boxed{
\text{Neighbors}
\rightarrow
\text{Reachability Distance}
\rightarrow
\text{LRD}
\rightarrow
\text{Compare LRDs}
\rightarrow
\text{LOF}
}
$$


# 7. Python Implementation — LOF

We can now implement the exact process we learned.

### Step 1: Create the data

```python
import numpy as np

X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [10]
])
```

Here, `10` is far from the main group.

---

### Step 2: Create the LOF model

```python
from sklearn.neighbors import LocalOutlierFactor

model = LocalOutlierFactor(
    n_neighbors=2,
    contamination=0.17
)
```

Here:

```text
n_neighbors = 2
    ↓
Use 2 nearest neighbors

contamination = 0.17
    ↓
Expect roughly 17% anomalies
```

For 6 observations:

$$
6\times0.17\approx1
$$

So approximately one point will be flagged.

---

### Step 3: Fit and predict

LOF's standard sklearn implementation performs detection using:

```python
predictions = model.fit_predict(X)
```

The output uses:

$$
+1=\text{Normal}
$$

$$
-1=\text{Anomaly}
$$

So you might get:

```text
[ 1  1  1  1  1 -1]
```

Meaning:

```text
1 → Normal
2 → Normal
3 → Normal
4 → Normal
5 → Normal
10 → Anomaly
```

---

### Step 4: Get the LOF scores

You can access the negative outlier factor using:

```python
scores = model.negative_outlier_factor_
```

**Important:** sklearn stores the **negative** of the LOF score.

So if conceptually:

$$
LOF(10)=3.67
$$

sklearn's value will be approximately:

$$
-3.67
$$

Therefore:

> **More negative `negative_outlier_factor_` → more anomalous.**

---

### Complete code

`Example below`

The important mapping is:

```text
fit_predict()
      ↓
+1 → Normal
-1 → Anomaly

negative_outlier_factor_
      ↓
More negative → More anomalous
```

One practical difference from Isolation Forest and One-Class SVM: **standard `LocalOutlierFactor` is primarily designed for outlier detection on the dataset you fit it on.** For detecting anomalies in completely new/unseen data, LOF has a separate `novelty=True` mode.


In [4]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [10]
])

model = LocalOutlierFactor(
    n_neighbors=2,
    contamination=0.17
)

predictions = model.fit_predict(X)
scores = model.negative_outlier_factor_

for point, score, prediction in zip(X, scores, predictions):
    print(
        point[0],
        score,
        prediction
    )

1 -1.249999999975 1
2 -1.249999999975 1
3 -0.6666666666888889 1
4 -1.249999999975 1
5 -1.249999999975 1
10 -3.6666666664888887 -1


### Question:

Why is standard `LocalOutlierFactor` mainly used on the dataset it was fitted on, and why do we need `novelty=True` for new/unseen data?

### Answer:

The key difference is **what LOF needs in order to calculate whether a point is an outlier**.

LOF is a **local** method. To decide whether point \(A\) is anomalous, it needs to:

1. Find \(A\)'s nearest neighbors.
2. Calculate \(A\)'s local density.
3. Calculate the local densities of \(A\)'s neighbors.
4. Compare them.

So LOF needs a **reference dataset around the point**.

---

### 1. Standard LOF

Suppose your training dataset is:

```text
1   2   3   4   5   10
```

You fit:

```python
model = LocalOutlierFactor(n_neighbors=2)
model.fit_predict(X)
```

LOF looks at the points **inside this dataset** and determines:

```text
1 → normal
2 → normal
3 → normal
4 → normal
5 → normal
10 → anomaly
```

Here, LOF is basically asking:

> "Among the points in this dataset, which ones have much lower local density than their neighbors?"

So standard LOF is designed for **outlier detection within the dataset you provide to it**.

---

### 2. What happens if a NEW point arrives?

Imagine tomorrow you receive:

```text
6
```

You want to ask:

> "Is 6 anomalous compared with my existing data?"

Conceptually, you want:

```text
Training data
1  2  3  4  5  10
          ↓
      learn/reference
          ↓
New point
     6
          ↓
    anomaly?
```

Now the situation is different.

You don't want LOF to treat `6` as another point in the original dataset and simply recalculate everything.

You want:

> **Keep the original dataset as the reference and evaluate the new point against it.**

That's what `novelty=True` is for.

---

### 3. Using `novelty=True`

```python
model = LocalOutlierFactor(
    n_neighbors=2,
    novelty=True
)

model.fit(X_train)
```

Then later:

```python
model.predict(X_new)
```

For example:

```python
X_train = [
    [1],
    [2],
    [3],
    [4],
    [5]
]
```

New data:

```python
X_new = [
    [6],
    [10]
]
```

The model can evaluate:

```text
Existing data
1  2  3  4  5
       ↓
   LOF reference
       ↓
New point: 6
       ↓
Is 6 unusual?

New point: 10
       ↓
Is 10 unusual?
```

---

### Why does this distinction matter?

Think of it this way:

| Mode                      | Question being asked                                               |
| ------------------------- | ------------------------------------------------------------------ |
| `novelty=False` (default) | "Which points in **this dataset** are outliers?"                   |
| `novelty=True`            | "Is this **new point** an outlier compared with my training data?" |

So:

**Standard LOF**

```text
Dataset → find outliers inside dataset
```

**LOF with `novelty=True`**

```text
Training dataset → establish reference
                         ↓
                    New point
                         ↓
                  anomaly detection
```

This is an important distinction because **LOF depends on local neighborhoods**, and those neighborhoods need to be defined consistently when evaluating unseen data.

Also, with `novelty=True`, you should use the fitted model to evaluate **new unseen data**, rather than using `fit_predict()` again on the same training set.


In [5]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor

# Training data
# These are considered normal examples
X_train = np.array([
    [1],
    [2],
    [3],
    [4],
    [5]
])

# Create LOF model
model = LocalOutlierFactor(
    n_neighbors=2,
    novelty=True
)

# Learn the local density structure
model.fit(X_train)

# New/unseen data
X_new = np.array([
    [4],
    [6],
    [10]
])

# Predict new points
predictions = model.predict(X_new)

# Get anomaly scores
scores = model.decision_function(X_new)

for point, prediction, score in zip(X_new, predictions, scores):
    if prediction == 1:
        result = "Normal"
    else:
        result = "Anomaly"

    print(
        f"Point: {point[0]}, "
        f"Score: {score:.3f}, "
        f"Result: {result}"
    )

Point: 4, Score: 0.667, Result: Normal
Point: 6, Score: 0.167, Result: Normal
Point: 10, Score: -2.167, Result: Anomaly


## 9. Important Hyperparameters — LOF

The main hyperparameters we need to understand are:

1. `n_neighbors`
2. `contamination`
3. `metric`
4. `algorithm`

Let's start with **`n_neighbors`** because it directly controls how LOF defines a point's local neighborhood.

### `n_neighbors`

```python
model = LocalOutlierFactor(n_neighbors=5)
```

This tells LOF:

> For each point, consider its **5 nearest neighbors** when calculating local density.

For example, suppose we have:

```text
1   2   3   4   5   10
```

If:

```python
n_neighbors = 2
```

then for point `10`, LOF looks at its closest 2 points:

```text
10 → 5, 4
```

So its local density is determined mainly from:

```text
10 ↔ 5
10 ↔ 4
```

---

### What if we increase `n_neighbors`?

Suppose:

```python
n_neighbors = 4
```

Now LOF considers:

```text
10 → 5, 4, 3, 2
```

So the neighborhood becomes larger.

Therefore:

```text
Small k
   ↓
Very local neighborhood
   ↓
More sensitive to small/local changes
```

while:

```text
Large k
   ↓
Larger neighborhood
   ↓
Broader view of density
```

### The important trade-off

| `n_neighbors` | Effect                                                  |
| ------------- | ------------------------------------------------------- |
| Small         | Very local, sensitive, but can be affected by noise     |
| Large         | More stable/broader, but may hide small local anomalies |

So `n_neighbors` controls **how local LOF is**.

It does **not** mean "number of clusters" or "number of anomalies."

For example:

```python
LocalOutlierFactor(n_neighbors=20)
```

means:

> Compare each point's local density with the densities around its 20 nearest neighbors.


### `contamination`

`contamination` tells LOF **how much of the dataset you expect to be anomalous**. It is mainly used to determine the final cutoff for deciding which points are labeled as anomalies.

```python
model = LocalOutlierFactor(
    n_neighbors=5,
    contamination=0.1
)
```

Here:

```text
contamination = 0.1
```

means you are saying:

> "I expect roughly 10% of the data to be anomalies."

For example, with **100 data points**:

```text
100 × 0.10 = 10
```

So roughly **10 points** may be classified as anomalies.

### Important distinction

`n_neighbors` and `contamination` do **different jobs**:

```text
n_neighbors
     ↓
How many nearby points should LOF compare with?
     ↓
Controls the LOCAL neighborhood


contamination
     ↓
How many points should ultimately be considered outliers?
     ↓
Controls the DECISION THRESHOLD
```

For example:

```python
LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)
```

means:

* Compare each point with its **20 nearest neighbors**
* Expect roughly **5%** of the data to be anomalous

If you increase contamination:

```text
0.01 → fewer points flagged
0.05 → more points flagged
0.10 → even more points flagged
```

### 3. `metric`

`metric` tells LOF **how to measure the distance between two points**.

```python
model = LocalOutlierFactor(
    n_neighbors=5,
    metric="euclidean"
)
```

The default is usually **Euclidean distance**.

For two points \(A\) and \(B\):

$$
d(A,B)=\sqrt{\sum_{i=1}^{m}(A_i-B_i)^2}
$$

For example:

```text
A = [1, 2]
B = [4, 6]
```

Distance:

$$
d(A,B)=\sqrt{(1-4)^2+(2-6)^2}
$$

$$
=\sqrt{9+16}
$$

$$
=5
$$

LOF uses these distances to determine the nearest neighbors and subsequently calculate density.

Common choices:

```python
metric="euclidean"
```

or

```python
metric="manhattan"
```

The choice matters because **different distance measures produce different neighborhoods**, which can change the LOF result.

---

### 4. `algorithm`

`algorithm` determines **how sklearn finds the nearest neighbors**.

```python
model = LocalOutlierFactor(
    n_neighbors=5,
    algorithm="auto"
)
```

Common options:

```text
"auto"
"ball_tree"
"kd_tree"
"brute"
```

#### `auto`

```python
algorithm="auto"
```

Usually the best choice when you don't have a specific reason to choose another method.

Sklearn decides an appropriate neighbor-search method.

#### `ball_tree`

Uses a **Ball Tree** structure to efficiently search for nearby points.

#### `kd_tree`

Uses a **KD Tree**, which can efficiently search neighbors in suitable low-dimensional datasets.

#### `brute`

Compares distances more directly between points.

This can become expensive for large datasets because many pairwise distances may need to be calculated.

So generally:

```python
algorithm="auto"
```

is a good default.

---

## 5. `leaf_size`

This is mainly relevant when using `ball_tree` or `kd_tree`.

```python
model = LocalOutlierFactor(
    n_neighbors=20,
    algorithm="ball_tree",
    leaf_size=30
)
```

`leaf_size` controls how many points are stored in the leaf nodes of the tree structure.

It mainly affects **speed and memory usage**, rather than changing the fundamental LOF idea.

For learning and most applications, you can usually leave the default.

---

## 6. `p`

`p` controls the Minkowski distance when using the corresponding distance metric.

For example:

```python
model = LocalOutlierFactor(
    n_neighbors=5,
    p=2
)
```

When:

$$
p=2
$$

the distance corresponds to **Euclidean distance**.

When:

$$
p=1
$$

it corresponds to **Manhattan distance**.

So:

```text
p = 1 → Manhattan
p = 2 → Euclidean
```

Again, this changes how "near" and "far" are defined, which can affect the neighbors and therefore the LOF score.

---

## LOF Hyperparameters — Final Picture

```text
                LOF
                 │
       ┌─────────┴─────────┐
       │                   │
n_neighbors          contamination
       │                   │
How many neighbors?   How many anomalies?
       │                   │
       └─────────┬─────────┘
                 │
          Local density
                 │
        ┌────────┴────────┐
        │                 │
      metric          algorithm
        │                 │
How is distance       How are nearest
calculated?           neighbors found?
        │
       p
        │
Which distance
behavior?
```

### The most important ones to remember

| Parameter       | Meaning                                            |
| --------------- | -------------------------------------------------- |
| `n_neighbors`   | Size of the local neighborhood                     |
| `contamination` | Expected proportion of anomalies / decision cutoff |
| `metric`        | How distance between points is measured            |
| `algorithm`     | How nearest neighbors are searched                 |
| `leaf_size`     | Tree-search performance parameter                  |
| `p`             | Controls Minkowski distance                        |

For most LOF applications, the parameters you'll care about most are **`n_neighbors` and `contamination`**. `metric` becomes important when the usual Euclidean distance isn't appropriate.
